In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
clients_raw = pd.read_csv(RAW / "clients.csv")
props_raw = pd.read_csv(RAW / "properties.csv")

print("clients:",clients_raw.shape)
print("properties:",props_raw.shape)

clients: (2000, 12)
properties: (10000, 9)


In [3]:
clients = clients_raw.copy()
props = props_raw.copy()

n_clients_start = len(clients)
n_props_start = len(props)

In [4]:
print(clients.dtypes, "\n")
print(props.dtypes, "\n")
print("clients nulls:\n", clients.isna().sum())
print("props nulls:\n", props.isna().sum())

client_id              object
client_type            object
first_name             object
last_name              object
date_of_birth          object
gender                 object
country                object
region                 object
acquisition_purpose    object
satisfaction_score      int64
loan_applied           object
referral_channel       object
dtype: object 

listing_id            int64
tower_number          int64
transaction_date     object
unit_category        object
unit_number           int64
floor_area_sqft     float64
sale_price           object
listing_status       object
client_ref           object
dtype: object 

clients nulls:
 client_id              0
client_type            0
first_name             0
last_name              0
date_of_birth          0
gender                 0
country                0
region                 0
acquisition_purpose    0
satisfaction_score     0
loan_applied           0
referral_channel       0
dtype: int64
props nulls:
 listing_id   

In [5]:
clients_cat_cols = ['client_type','gender','country','region',
                   'acquisition_purpose','loan_applied','referral_channel']
props_cat_cols = ['unit_category','listing_status']

In [6]:
for col in clients_cat_cols:
    print(f"---{col}---")
    print(clients[col].value_counts(dropna=False))
    print()

---client_type---
client_type
Individual    1897
Company        103
Name: count, dtype: int64

---gender---
gender
M    1012
F     988
Name: count, dtype: int64

---country---
country
USA          1538
UK             95
Canada         85
Germany        56
France         53
Belgium        43
Mexico         40
Australia      39
Russia         36
Denmark        15
Name: count, dtype: int64

---region---
region
California                633
Nevada                    143
Colorado                  118
Arizona                   108
Oregon                     96
Utah                       87
Washington                 73
Virginia                   65
Texas                      56
Florida                    50
New York                   49
Georgia                    34
England                    29
Alberta                    27
Scotland                   25
Ohio                       24
Northern Ireland           24
British Columbia           20
Wales                      17
Brittany           

In [7]:
for col in props_cat_cols:
    print(f"---{col}---")
    print(props[col].value_counts(dropna=False))
    print()

---unit_category---
unit_category
Apartment    8547
Office       1453
Name: count, dtype: int64

---listing_status---
listing_status
Sold         7305
Available    2695
Name: count, dtype: int64



In [8]:
print(pd.crosstab(props['listing_status'], props['client_ref'].isna()))

client_ref      False  True 
listing_status              
Available           0   2695
Sold             7305      0


client_ref is null only for Available listings (2695)

In [9]:
print(props.groupby("listing_status")["sale_price"].apply(lambda s: s.head(5).tolist()))

listing_status
Available    [$216,826.00, $237,680.88, $166,623.12, $233,3...
Sold         [$300,385.62, $208,930.81, $218,585.92, $246,1...
Name: sale_price, dtype: object


In [10]:
price_num = pd.to_numeric(props['sale_price'], errors="coerce")

print("Unparseable as number:", price_num.isna().sum())
print()
print(props.assign(price_num=price_num)
        .groupby('listing_status')['price_num']
        .agg(rows="size",zero=lambda s:(s==0).sum(),min="min", median="median", max="max"))

Unparseable as number: 10000

                rows  zero  min  median  max
listing_status                              
Available       2695     0  NaN     NaN  NaN
Sold            7305     0  NaN     NaN  NaN


In [11]:
props['sale_price'] = (
    props['sale_price']
    .str.replace("$","", regex=False)
    .str.replace(",","", regex=False)
    .astype(float)
)

In [12]:
print(props['sale_price'].dtype)
print(props.groupby('listing_status')['sale_price'].agg(["size","min","median","max"]))

float64
                size       min     median        max
listing_status                                      
Available       2695  97537.58  329697.10  729836.68
Sold            7305  97402.80  330997.32  736652.27


In [13]:
print(props['transaction_date'].head(10).tolist())
print(clients['date_of_birth'].head(10).tolist())

['01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024', '01-01-2024']
['05-11-1968', '11/26/1962', '04-07-1959', '11/25/1959', '2/28/1976', '03-06-1957', '5/24/1947', '10/17/1969', '10-05-1975', '6/17/1966']


In [14]:
dash = clients.loc[clients['date_of_birth'].str.contains("-"),"date_of_birth"]
parts = dash.str.split("-",expand=True).astype(int)

print("dash-format rows:", len(dash))
print("first number > 12:",(parts[0]>12).sum())
print("second number > 12:",(parts[1]>12).sum())

dash-format rows: 855
first number > 12: 0
second number > 12: 0


In [15]:
td = props['transaction_date']
print("slash rows:", td.str.contains("/").sum(), "dash rows:", td.str.contains("-").sum())

tparts = td[td.str.contains("-")].str.split("-", expand=True).astype(int)
print("first number > 12:", (tparts[0]>12).sum())
print("second number > 12:",(tparts[1]>12).sum())
print("unique years:", sorted(tparts[2].unique()))

slash rows: 0 dash rows: 10000
first number > 12: 0
second number > 12: 0
unique years: [np.int64(2024), np.int64(2025)]


In [16]:
print("first number values:", sorted(tparts[0].unique()))
print("second number values:", sorted(tparts[1].unique()))

first number values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
second number values: [np.int64(1)]


In [17]:
dob = clients_raw["date_of_birth"]
is_slash = dob.str.contains("/")

clients["date_of_birth"] = pd.NaT
clients.loc[is_slash, "date_of_birth"] = pd.to_datetime(dob[is_slash], format="%m/%d/%Y", errors="coerce")
clients.loc[~is_slash, "date_of_birth"] = pd.to_datetime(dob[~is_slash], format="%m-%d-%Y", errors="coerce")
clients["date_of_birth"] = pd.to_datetime(clients["date_of_birth"])

print(clients["date_of_birth"].isna().sum(), "failed to parse")
print(clients["date_of_birth"].min(), clients["date_of_birth"].max())

0 failed to parse
1931-02-13 00:00:00 2000-12-19 00:00:00


In [18]:
print("clients: exact duplicate rows:", clients.duplicated().sum())
print("clients: duplicate client_id:", clients['client_id'].duplicated().sum())
print("clients: exact duplicate rows:", props.duplicated().sum())
print("clients: duplicate listing_id:", props['listing_id'].duplicated().sum())

clients: exact duplicate rows: 0
clients: duplicate client_id: 0
clients: exact duplicate rows: 0
clients: duplicate listing_id: 0


In [19]:
sold = props[props['client_ref'].notna()]
orphans = ~sold['client_ref'].isin(clients['client_id'])

print("sold rows:", len(sold))
print("client_ref not found in clients:", orphans.sum())
print("client with no purchase:", (~clients['client_id'].isin(sold['client_ref'])).sum())
print("avg purchase per buying client:", round(len(sold)/ sold['client_ref'].nunique(),2))

sold rows: 7305
client_ref not found in clients: 0
client with no purchase: 0
avg purchase per buying client: 3.65


In [20]:
print(clients['satisfaction_score'].describe())
print(props[['tower_number','unit_number','floor_area_sqft']].describe())

count    2000.000000
mean        3.029000
std         1.413562
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: satisfaction_score, dtype: float64
       tower_number  unit_number  floor_area_sqft
count  10000.000000  10000.00000     10000.000000
mean      10.370100     28.99920      1139.941412
std        5.769025     16.88242       418.373967
min        1.000000      1.00000       410.710000
25%        5.000000     15.00000       782.200000
50%       10.000000     29.00000      1110.880000
75%       15.000000     43.00000      1499.000000
max       20.000000     70.00000      1957.160000


In [21]:
props['transaction_date'] = pd.to_datetime(props_raw['transaction_date'], format = "%m-%d-%Y", errors="coerce")
print(props['transaction_date'].isna().sum(),"failed to parse")
print(props['transaction_date'].min(), props['transaction_date'].max())
print(props['transaction_date'].dt.to_period("M").value_counts().sort_index())

0 failed to parse
2024-01-01 00:00:00 2025-12-01 00:00:00
transaction_date
2024-01    502
2024-02    510
2024-03    508
2024-04    518
2024-05    506
2024-06    537
2024-07    517
2024-08    480
2024-09    455
2024-10    490
2024-11    494
2024-12    497
2025-01    326
2025-02    339
2025-03    320
2025-04    332
2025-05    339
2025-06    325
2025-07    368
2025-08    338
2025-09    320
2025-10    344
2025-11    318
2025-12    317
Freq: M, Name: count, dtype: int64


In [22]:
print(clients[clients['client_type'] == "company"].head(10))
print(clients.groupby('client_type')['acquisition_purpose'].value_counts(normalize=True))

Empty DataFrame
Columns: [client_id, client_type, first_name, last_name, date_of_birth, gender, country, region, acquisition_purpose, satisfaction_score, loan_applied, referral_channel]
Index: []
client_type  acquisition_purpose
Company      Home                   0.650485
             Investment             0.349515
Individual   Home                   0.694781
             Investment             0.305219
Name: proportion, dtype: float64


In [23]:
companies = clients[clients["client_type"].str.contains("Comp", case=False)]
print(len(companies))
print(companies.head(10))

103
    client_id client_type   first_name last_name date_of_birth gender country  \
4       C0005     Company      Marleez        Co    1976-02-28      M     USA   
11      C0012     Company        Kylax        Co    1975-04-25      M     USA   
13      C0014     Company       Esther        Co    1980-07-10      M     USA   
53      C0054     Company  Bridger CAL        Co    1967-02-15      M     USA   
57      C0058     Company       Kenyon        Co    1969-05-03      F     USA   
127     C0128     Company       Abdiel        Co    1963-05-22      M     USA   
154     C0155     Company         Kamd        Co    1975-08-18      M     USA   
220     C0221     Company        Atlas        Co    1987-05-14      F     USA   
229     C0230     Company      Forsyte        Co    1977-07-22      M     USA   
260     C0261     Company       Celera        Co    1985-05-13      F     USA   

         region acquisition_purpose  satisfaction_score loan_applied  \
4    California          Investm

In [24]:
clients['is_company'] = (clients['client_type'] == 'Company').astype(int)

sold = props[props['client_ref'].notna()]
per_client = sold.groupby('client_ref').agg(n_bought=("listing_id","size"),
                                           total_spend=("sale_price","sum"))
check = clients.merge(per_client, left_on="client_id", right_index=True)

print(check.groupby('client_type')[["n_bought","total_spend"]].mean().round(0))
print(clients[clients['client_type'] == "Company"]['country'].value_counts())

             n_bought  total_spend
client_type                       
Company           4.0    1263753.0
Individual        4.0    1260192.0
country
USA          80
France        5
UK            5
Germany       3
Mexico        3
Canada        3
Australia     2
Belgium       1
Russia        1
Name: count, dtype: int64


103 company clients retained thier gender and date_of_birth are placeholder values.Purchase behaviour (4 properties, about $1.26M spend)
matches individuals,so they are ketp and flagged with is_company.


In [25]:
print('clients:', n_clients_start,"->",len(clients))
print('props:', n_props_start,"->",len(props))
print(clients.dtypes)
print(props.dtypes)

clients: 2000 -> 2000
props: 10000 -> 10000
client_id                      object
client_type                    object
first_name                     object
last_name                      object
date_of_birth          datetime64[ns]
gender                         object
country                        object
region                         object
acquisition_purpose            object
satisfaction_score              int64
loan_applied                   object
referral_channel               object
is_company                      int64
dtype: object
listing_id                   int64
tower_number                 int64
transaction_date    datetime64[ns]
unit_category               object
unit_number                  int64
floor_area_sqft            float64
sale_price                 float64
listing_status              object
client_ref                  object
dtype: object


In [26]:
clients.to_csv(PROCESSED / "clients_clean.csv", index=False)
props.to_csv(PROCESSED / "props_clean.csv", index=False)